# Microbiome & IBD Study — Step 1: Exploratory Data Analysis

**Dataset:** HMP2 / IBDMDB — Lloyd-Price et al., *Nature* (2019)  
**Goal:** Understand the structure, distributions, and biological patterns in the microbiome data before modelling.

We have **3,387 stool samples** from 132 individuals diagnosed with:
- **Crohn's Disease (CD)**
- **Ulcerative Colitis (UC)**
- **Healthy controls**

Each sample contains relative abundances of **566 bacterial species**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

# Load data
df = pd.read_csv('../data/Microbiome_Metagenomics.csv')
print(f'Dataset shape: {df.shape}')
df.head(3)

## 1.1 Dataset Overview

In [ ]:
# Separate metadata from bacterial abundance columns
meta_cols = ['External ID', 'Participant ID', 'week_num', 'diagnosis', 'fecalcal']
bacteria_cols = [c for c in df.columns if c not in meta_cols]

print(f'Metadata columns : {len(meta_cols)}')
print(f'Bacterial species: {len(bacteria_cols)}')
print(f'Total samples    : {len(df)}')
print(f'Unique patients  : {df["Participant ID"].nunique()}')
print(f'\nDiagnosis counts:')
print(df['diagnosis'].value_counts())
print(f'\nMissing values per metadata column:')
print(df[meta_cols].isna().sum())

**Key observations:**
- `fecalcal` (fecal calprotectin, a gold-standard inflammation biomarker) has ~60% missing values — it will **not** be used as a feature, but we will use it later to biologically validate our XAI findings.
- The dataset is **longitudinal**: multiple samples per patient across weeks 0–57.
- The target `diagnosis` is **imbalanced**: Crohn's (1608) > UC (913) ≈ Healthy (866).

## 1.2 Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['diagnosis'].value_counts()
colors = ['#e07b54', '#5b8db8', '#6aab6a']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Sample Count per Diagnosis', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of samples')
for i, (label, val) in enumerate(counts.items()):
    axes[0].text(i, val + 20, str(val), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportions', fontsize=13, fontweight='bold')

plt.suptitle('Class Imbalance in IBD Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/01_class_distribution.png', bbox_inches='tight')
plt.show()
print('Imbalance ratio (Crohns/Healthy):', round(counts["Crohns Disease"] / counts["Healthy"], 2))

## 1.3 Longitudinal Structure — Samples per Patient Over Time

In [ ]:
# Samples per patient
samples_per_patient = df.groupby(['Participant ID', 'diagnosis']).size().reset_index(name='n_samples')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution of samples per patient
sns.boxplot(data=samples_per_patient, x='diagnosis', y='n_samples',
            palette=['#e07b54', '#5b8db8', '#6aab6a'], ax=axes[0])
axes[0].set_title('Samples per Patient by Diagnosis', fontweight='bold')
axes[0].set_ylabel('Number of samples')
axes[0].set_xlabel('')

# Week distribution
sns.histplot(data=df, x='week_num', hue='diagnosis',
             palette=['#e07b54', '#5b8db8', '#6aab6a'],
             multiple='stack', bins=30, ax=axes[1])
axes[1].set_title('Sample Collection by Study Week', fontweight='bold')
axes[1].set_xlabel('Study week')

plt.tight_layout()
plt.savefig('../figures/02_longitudinal_structure.png', bbox_inches='tight')
plt.show()

## 1.4 Bacterial Abundance Distribution

Microbiome data is typically **sparse** (many zeros) and **right-skewed** — most bacteria are absent or rare in most samples.

In [ ]:
# Sparsity
zero_pct = (df[bacteria_cols] == 0).sum().sum() / (df[bacteria_cols].shape[0] * df[bacteria_cols].shape[1]) * 100
print(f'Sparsity (% zeros across all bacteria): {zero_pct:.1f}%')

# Distribution of a dominant species
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Overall abundance distribution (log scale)
mean_abundances = df[bacteria_cols].mean().sort_values(ascending=False)
axes[0].plot(range(len(mean_abundances)), mean_abundances.values, color='#5b8db8')
axes[0].set_yscale('log')
axes[0].set_title('Mean Abundance Rank (all 566 species)', fontweight='bold')
axes[0].set_xlabel('Rank')
axes[0].set_ylabel('Mean relative abundance (log scale)')

# Histogram of Faecalibacterium prausnitzii (key health-associated species)
axes[1].hist(df['Faecalibacterium prausnitzii'] + 1e-6, bins=50,
             color='#6aab6a', edgecolor='white')
axes[1].set_title('Faecalibacterium prausnitzii Abundance\n(key anti-inflammatory species)', fontweight='bold')
axes[1].set_xlabel('Relative abundance')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../figures/03_abundance_distribution.png', bbox_inches='tight')
plt.show()

## 1.5 Top 15 Most Abundant Bacteria by Diagnosis

In [ ]:
top15 = df[bacteria_cols].mean().sort_values(ascending=False).head(15).index.tolist()

# Mean abundance per diagnosis
top_by_diag = df.groupby('diagnosis')[top15].mean().T

fig, ax = plt.subplots(figsize=(13, 6))
top_by_diag.plot(kind='bar', ax=ax,
                 color=['#e07b54', '#6aab6a', '#5b8db8'],
                 edgecolor='white', width=0.75)
ax.set_title('Top 15 Bacteria — Mean Relative Abundance by Diagnosis', fontsize=13, fontweight='bold')
ax.set_ylabel('Mean relative abundance')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=45)
plt.xticks(ha='right', fontsize=9)
plt.legend(title='Diagnosis')
plt.tight_layout()
plt.savefig('../figures/04_top15_by_diagnosis.png', bbox_inches='tight')
plt.show()

## 1.6 PCA — Can Diagnosis Groups Be Separated?

We use PCA (Principal Component Analysis) to project the 566-dimensional bacterial space into 2D. This gives us a first visual sense of whether the three diagnosis groups form separable clusters.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Fill NaNs with 0 (absent species) for PCA only
X_pca = df[bacteria_cols].fillna(0).values
X_scaled = StandardScaler().fit_transform(X_pca)

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({'PC1': coords[:, 0], 'PC2': coords[:, 1], 'diagnosis': df['diagnosis']})

fig, ax = plt.subplots(figsize=(9, 6))
palette = {'Crohns Disease': '#e07b54', 'Ulcerative Colitis': '#5b8db8', 'Healthy': '#6aab6a'}

for diag, grp in pca_df.groupby('diagnosis'):
    ax.scatter(grp['PC1'], grp['PC2'], label=diag,
               color=palette[diag], alpha=0.35, s=15, edgecolors='none')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
ax.set_title('PCA of Microbiome Profiles by Diagnosis\n(566 bacterial species → 2 components)', fontsize=13, fontweight='bold')
ax.legend(title='Diagnosis', markerscale=2)
plt.tight_layout()
plt.savefig('../figures/05_pca.png', bbox_inches='tight')
plt.show()

print(f'Total variance explained by 2 PCs: {sum(pca.explained_variance_ratio_)*100:.1f}%')

## 1.7 Correlation Heatmap — Top 20 Bacteria

Do highly abundant bacteria co-occur or exclude each other? This reveals ecological relationships in the gut.

In [ ]:
top20 = df[bacteria_cols].mean().sort_values(ascending=False).head(20).index.tolist()
corr = df[top20].corr()

# Shorten names for readability
short_names = {c: c.split(' ')[0][0] + '. ' + c.split(' ')[1] if len(c.split(' ')) > 1 else c for c in top20}
corr_renamed = corr.rename(index=short_names, columns=short_names)

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_renamed, dtype=bool))
sns.heatmap(corr_renamed, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={'size': 7})
ax.set_title('Spearman Correlation — Top 20 Bacterial Species', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/06_correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 1.8 EDA Summary

| Finding | Implication for modelling |
|---|---|
| 3,387 samples × 566 species | High-dimensional; feature selection may help |
| Class imbalance (CD 47%, UC 27%, H 26%) | Must apply SMOTE or class weights |
| ~60% NaN in `fecalcal` | Drop as feature; use for biological validation |
| High sparsity in bacteria abundances | Log/CLR transformation recommended |
| PCA shows partial separation | Non-linear models (XGBoost) likely better than linear |
| *F. prausnitzii* lower in IBD | Matches known biology — good sanity check |